In [ ]:
import base64, glob, hashlib, io, os, shlex, subprocess, tarfile
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '934900f9ed223d117d219d5fa232b0ee2f7f3a83cdbb3d86fbf6b725757719d3'
CODE, OUT = '/kaggle/working/code', '/kaggle/working/out'
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall(CODE)
print('code sha256 934900f9ed223d117d219d5fa232b0ee2f7f3a83cdbb3d86fbf6b725757719d3 from git fc2f350+local changes', flush=True)
subprocess.run('nproc; free -g | head -2', shell=True)


def find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True), key=len)
    return hits[0] if hits else None


chik = [p for p in glob.glob('/kaggle/input/**/*.mat', recursive=True) if 'chikusei' in p.lower() and 'Ground' not in p]
DATA = dict(chikusei=sorted(chik, key=len)[0] if chik else None, pavia=find('Pavia.mat'),
            cave=os.path.dirname(os.path.dirname(os.path.dirname(find('Train/HSI/balloons.mat') or '/x/x/x/x'))),
            harvard=os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(find('Data/Train/HSI/img*.mat') or '/x/x/x/x/x')))),
            neon='/kaggle/input')
CACHE = dict(chikusei='/tmp/chikusei.npy', harvard='/tmp/harvard')
JOBS = [('cond-chikusei-x4-corng', "--dataset chikusei --scale 4 --model puformer --pad reflect --omc 1 --dstep kronlearn --mres 1 --sa 1 --only 'nominal,MSI local,HSI stripes,MSI changes,HSI band'", 'chikusei'), ('cond-pavia-x4-corng', "--dataset pavia --scale 4 --model puformer --pad reflect --omc 1 --dstep kronlearn --mres 1 --sa 1 --only 'nominal,MSI local,HSI stripes,MSI changes,HSI band'", 'pavia')]
for name, rargs, ds in JOBS:
    ck = find(f'{name}/best_ema.pt')
    if not ck:
        print('no checkpoint for', name, flush=True); continue
    cache = f'--cache {CACHE[ds]}' if ds in CACHE else ''
    out = f'{OUT}/{name}'
    extra = f'--train_dir {os.path.dirname(ck)}' if 'cond_eval.py' == 'test.py' else ''
    cmd = f"python cond_eval.py --mat {shlex.quote(DATA[ds])} {cache} {rargs} --ckpt {ck} --out {out} {extra}"
    print('RUN', name, ':', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=CODE)
    print('END', name, 'rc', r.returncode, flush=True)
